# 02 - Preprocessing Pipeline

This notebook documents the preprocessing methodology used for CarePilot priority prediction.

It demonstrates text normalization, punctuation cleaning, stopword removal, TF-IDF preparation, categorical encoding, missing value handling, and deterministic train/validation/test splitting.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / "ml").exists():
    ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "ml").exists())

BENCHMARK_DIR = ROOT / "ml" / "benchmarks" / "transfer"
if str(BENCHMARK_DIR) not in sys.path:
    sys.path.append(str(BENCHMARK_DIR))

ARTIFACTS_DIR = ROOT / "ml" / "artifacts"
PLOTS_DIR = ROOT / "ml" / "plots"
REPORTS_DIR = ROOT / "ml" / "reports"
for directory in [ARTIFACTS_DIR, PLOTS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

from preprocessing import (
    TARGET_COLUMN,
    TEXT_COLUMN,
    build_preprocessor,
    clean_text,
    feature_matrix,
    load_database_url,
    load_priority_dataset,
    prepare_dataframe,
)

In [2]:
raw_df = load_priority_dataset(load_database_url(ROOT))
df = prepare_dataframe(raw_df)

print(f"Loaded rows: {len(df):,}")
display(raw_df[["ticket_id", "title", "description", "perimeter", "priority", "sla_breached", "resolution_time_minutes", "incident_category", "priority"]].head(6))

Loaded rows: 14,000


,ticket_id,title,description,perimeter,priority,sla_breached,resolution_time_minutes,incident_category,priority
0,ML-CARE-000001,coupon engine timeout,coupon engine timeout - customer discount not ...,Loyalty,Medium,False,NaN,COUPON_VALIDATION_TIMEOUT,False
1,ML-CARE-000002,dns resolution failure failure at store,checkout subnet affected at Carrefour Market P...,LAN/WAN,Critical,True,797.0,DNS_RESOLUTION_FAILURE,True
2,ML-CARE-000003,POS lane down,POS lane down - cashier cannot process baskets...,Checkout (VLP),Low,False,1634.0,CHECKOUT_LANE_UNAVAILABLE,False
3,ML-CARE-000004,backup backup job failed - store systems,None,local server,High,False,NaN,BACKUP_JOB_FAILED,True
4,ML-CARE-000005,PIN pad encryption error,key mismatch; impact reported by store ops. Ne...,Payment,Medium,False,1767.0,PINPAD_ENCRYPTION_ERROR,False
5,ML-CARE-000006,Store server unreachable,Store server unreachable lanes in degraded mod...,Store Infrastructure,Low,False,1951.0,STORE_SERVER_UNREACHABLE,False


## Text Normalization

The pipeline lowercases operational text and removes punctuation while preserving numbers such as checkout lane IDs, device numbers, and store references.

In [3]:
examples = raw_df.loc[raw_df["description"].fillna("").str.len() > 0, ["title", "description"]].head(8).copy()
examples["raw_text"] = examples["title"].fillna("") + " " + examples["description"].fillna("")
examples["normalized_text"] = examples["raw_text"].map(clean_text)
examples[["raw_text", "normalized_text"]]

,raw_text,normalized_text
0,coupon engine timeout coupon engine timeout - ...,coupon engine timeout coupon engine timeout cu...
1,dns resolution failure failure at store checko...,dns resolution failure failure at store checko...
2,POS lane down POS lane down - cashier cannot p...,pos lane down pos lane down cashier cannot pro...
4,PIN pad encryption error key mismatch; impact ...,pin pad encryption error key mismatch impact r...
5,Store server unreachable Store server unreacha...,store server unreachable store server unreacha...
6,receipt printer offline receipt printer offlin...,receipt printer offline receipt printer offlin...
7,database volume critical queries slowing. Vend...,database volume critical queries slowing vendo...
8,COUPON REDEMPTION SERVICE TIMEOUT customer dis...,coupon redemption service timeout customer dis...


## Stopword Removal and TF-IDF Preparation

Stopword removal is performed by `TfidfVectorizer(stop_words="english")`. The benchmark uses unigrams and bigrams to capture both individual operational keywords and short incident phrases.

In [4]:
def remove_stopwords(text: str) -> str:
    return " ".join(token for token in text.split() if token not in ENGLISH_STOP_WORDS)

sample_text = examples["normalized_text"].iloc[0]
print("Before stopword filtering:")
print(sample_text)
print("\nAfter stopword filtering:")
print(remove_stopwords(sample_text))

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_df=0.92, max_features=3500)
tfidf_preview = vectorizer.fit_transform(df[TEXT_COLUMN])
print(f"TF-IDF matrix shape: {tfidf_preview.shape}")
print("Sample TF-IDF features:", vectorizer.get_feature_names_out()[:25].tolist())

Before stopword filtering:
coupon engine timeout coupon engine timeout customer discount not applied store says says issue is intermittent and affects customer flow

After stopword filtering:
coupon engine timeout coupon engine timeout customer discount applied store says says issue intermittent affects customer flow


TF-IDF matrix shape: (14000, 2026)
Sample TF-IDF features: ['absent', 'absent device', 'account', 'account retrieved', 'acquirer', 'acquirer responses', 'affected', 'affected carrefour', 'affected checkout', 'affected chkout', 'affected impact', 'affected ip', 'affected local', 'affected loyalty', 'affected service', 'affected store', 'affected vendor', 'affects', 'affects affects', 'affects customer', 'agent', 'agent offline', 'alert', 'alert capacity', 'alert database']


## Missing Values and Categorical Encoding

Text missingness is converted to an empty string. Numeric missing values are imputed with the median. Categorical fields are one-hot encoded with unknown categories ignored at inference time.

In [5]:
missing_before = raw_df.isna().sum().sort_values(ascending=False)
print("Missing values before preprocessing:")
display(missing_before.to_frame("missing_count"))

preprocessor = build_preprocessor(max_text_features=3500)
X, y = feature_matrix(df)
X_preview = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

print(f"Prepared matrix shape: {X_preview.shape}")
print(f"Number of encoded features: {len(feature_names):,}")
print("Example encoded feature names:")
print(feature_names[:30].tolist())

Missing values before preprocessing:


,missing_count
resolution_time_minutes,3644
description,1134
ticket_id,0
perimeter,0
title,0
priority,0
sla_breached,0
incident_category,0
priority,0


Prepared matrix shape: (14000, 2085)
Number of encoded features: 2,085
Example encoded feature names:
['text__absent', 'text__absent device', 'text__account', 'text__account retrieved', 'text__acquirer', 'text__acquirer responses', 'text__affected', 'text__affected carrefour', 'text__affected checkout', 'text__affected chkout', 'text__affected impact', 'text__affected ip', 'text__affected local', 'text__affected loyalty', 'text__affected service', 'text__affected store', 'text__affected vendor', 'text__affects', 'text__affects affects', 'text__affects customer', 'text__agent', 'text__agent offline', 'text__alert', 'text__alert capacity', 'text__alert database', 'text__alert local', 'text__alert queries', 'text__alert raised', 'text__alerts', 'text__alerts attendant']


## Deterministic Train / Validation / Test Split

The split is stratified on `priority` to preserve the target distribution across train, validation, and test subsets.

In [6]:
RANDOM_STATE = 20260525
train_df, holdout_df = train_test_split(
    df,
    train_size=0.70,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)
validation_df, test_df = train_test_split(
    holdout_df,
    train_size=0.50,
    random_state=RANDOM_STATE,
    stratify=holdout_df[TARGET_COLUMN],
)

split_summary = pd.DataFrame({
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "positive_rate": [train_df[TARGET_COLUMN].mean(), validation_df[TARGET_COLUMN].mean(), test_df[TARGET_COLUMN].mean()],
}, index=["train", "validation", "test"])
split_summary["positive_rate"] = (split_summary["positive_rate"] * 100).round(2)
display(split_summary)

split_payload = split_summary.reset_index(names="split").to_dict(orient="records")
(ARTIFACTS_DIR / "02_preprocessing_split_summary.json").write_text(json.dumps(split_payload, indent=2), encoding="utf-8")

,rows,positive_rate
train,9800,40.0
validation,2100,40.0
test,2100,40.0


231

## Preprocessing Summary

This preprocessing design is intentionally classical and reproducible. It supports academic benchmarking with Logistic Regression, Random Forest, and XGBoost without introducing deep learning, transformers, or production-facing AI behavior.